In [1]:
%pip install python-crfsuite
import nltk
nltk.download('conll2002')
from nltk.corpus import conll2002

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package conll2002 to
[nltk_data]     C:\Users\bsanc\AppData\Roaming\nltk_data...
[nltk_data]   Package conll2002 is already up-to-date!


In [19]:
import os

models_dir = os.path.join(os.getcwd(), 'models')
os.makedirs(models_dir, exist_ok=True)

In [2]:
esp_train = conll2002.iob_sents('esp.train') # Train, ned.train => Neerlandès
esp_val = conll2002.iob_sents('esp.testa') # Dev
esp_test = conll2002.iob_sents('esp.testb') # Test

In [3]:
ned_train = conll2002.iob_sents('ned.train') # Train, ned.train => Neerlandès
ned_val = conll2002.iob_sents('ned.testa') # Dev
ned_test = conll2002.iob_sents('ned.testb')

In [4]:
def prepare_data(sents):
    return [[(word,bio) for word, pos, bio in sent] for sent in sents]

In [5]:
esp_train_net = prepare_data(esp_train)
esp_val_net = prepare_data(esp_val)
esp_test_net = prepare_data(esp_test)
ned_train_net = prepare_data(ned_train)
ned_val_net = prepare_data(ned_val)
ned_test_net = prepare_data(ned_test)

In [6]:
def bio_to_io(sentence):
    new_sentence = []
    
    for word, tag in sentence:
        if tag.startswith("B-") or tag.startswith("I-"):
            new_tag = "I-" + tag[2:]
        else:
            new_tag = tag  # "O"
            
        new_sentence.append((word, new_tag))
    
    return new_sentence

In [7]:
def bio_to_biow(sentence):
    new_sentence = []
    i = 0
    n = len(sentence)
    
    while i < n:
        word, tag = sentence[i]
        
        if tag.startswith("B-"):
            entity_type = tag[2:]
            j = i + 1
            
            # trobar final de l'entitat
            while j < n and sentence[j][1] == f"I-{entity_type}":
                j += 1
            
            length = j - i
            
            if length == 1:
                new_sentence.append((word, f"W-{entity_type}"))
            else:
                new_sentence.append((word, f"B-{entity_type}"))
                for k in range(i+1, j):
                    new_sentence.append((sentence[k][0], f"I-{entity_type}"))
            
            i = j
        
        else:
            new_sentence.append((word, tag))
            i += 1
    
    return new_sentence

In [8]:
def bio_to_bioew(sentence):
    new_sentence = []
    i = 0
    n = len(sentence)
    
    while i < n:
        word, tag = sentence[i]
        
        if tag.startswith("B-"):
            entity_type = tag[2:]
            j = i + 1
            
            # trobar final de l'entitat
            while j < n and sentence[j][1] == f"I-{entity_type}":
                j += 1
            
            length = j - i
            
            if length == 1:
                new_sentence.append((word, f"W-{entity_type}"))
            else:
                # primer
                new_sentence.append((word, f"B-{entity_type}"))
                
                # mig
                for k in range(i+1, j-1):
                    new_sentence.append((sentence[k][0], f"I-{entity_type}"))
                
                # últim
                last_word = sentence[j-1][0]
                new_sentence.append((last_word, f"E-{entity_type}"))
            
            i = j
        
        else:
            new_sentence.append((word, tag))
            i += 1
    
    return new_sentence

In [13]:
from typing import List
import string

class FeatureFunc:
    def __init__(self, use_casing=True, use_punct=True, use_nums=True, 
                 use_suffix=True, use_prefix=True, use_length=True, 
                 use_context=True, use_pos=True):
        self.use_casing  = use_casing
        self.use_punct   = use_punct
        self.use_nums    = use_nums
        self.use_suffix  = use_suffix
        self.use_prefix  = use_prefix
        self.use_length  = use_length
        self.use_context = use_context
        self.use_pos     = use_pos

    def __call__(self, tokens: List[str], idx: int) -> List[str]:
        word = tokens[idx]

        # --- OBLIGATÒRIES ---

        # Paraula actual (sempre)
        features = ['word=' + word, 'lower=' + word.lower()]

        # És majúscula?
        if self.use_casing:
            features.append('isupper=%s'  % word.isupper())
            features.append('istitle=%s'  % word.istitle())
            features.append('islower=%s'  % word.islower())

        # Té signes de puntuació?
        if self.use_punct:
            features.append('ispunct=%s'   % all(c in string.punctuation for c in word))
            features.append('haspunct=%s'  % any(c in string.punctuation for c in word))

        # Té números?
        if self.use_nums:
            features.append('isdigit=%s'   % word.isdigit())
            features.append('hasdigit=%s'  % any(c.isdigit() for c in word))

        # Sufixos
        if self.use_suffix:
            features.append('suffix1=' + word[-1:])
            features.append('suffix2=' + word[-2:])
            features.append('suffix3=' + word[-3:])

        # --- OPCIONALS ---

        # Prefixos
        if self.use_prefix:
            features.append('prefix1=' + word[:1])
            features.append('prefix2=' + word[:2])
            features.append('prefix3=' + word[:3])

        # Longitud
        if self.use_length:
            features.append('len=%d' % len(word))

        # Context (paraula anterior i següent)
        if self.use_context:
            if idx > 0:
                prev = tokens[idx-1]
                features.append('prev='        + prev.lower())
                features.append('prev_istitle=%s' % prev.istitle())
                features.append('prev_isupper=%s' % prev.isupper())
            else:
                features.append('BOS')
            if idx < len(tokens) - 1:
                next_ = tokens[idx+1]
                features.append('next='        + next_.lower())
                features.append('next_istitle=%s' % next_.istitle())
                features.append('next_isupper=%s' % next_.isupper())
            else:
                features.append('EOS')

        return features

In [10]:
def convert_coding(dataset, coding):
    if coding == "BIO":
        return dataset
    elif coding == "IO":
        return [bio_to_io(sent) for sent in dataset]
    elif coding == "BIOW":
        return [bio_to_biow(sent) for sent in dataset]
    elif coding == "BIOEW":
        return [bio_to_bioew(sent) for sent in dataset]
    else:
        raise ValueError(f"Coding desconegut: {coding}")

In [11]:
def evaluate(model, data):
    tp, fp, fn = 0, 0, 0

    for sent in data:
        tokens  = [word for word, tag in sent]
        gold    = [tag  for word, tag in sent]
        pred    = [tag  for word, tag in model.tag(tokens)]

        for g, p in zip(gold, pred):
            g_is_entity = g != 'O'
            p_is_entity = p != 'O'

            if g_is_entity and p_is_entity:
                if g == p:
                    tp += 1   # mateixa etiqueta (ex: B-PER == B-PER)
                else:
                    fp += 1   # entitat però etiqueta diferent (ex: B-PER vs I-PER)
                    fn += 1   # la gold no s'ha trobat correctament
            elif p_is_entity and not g_is_entity:
                fp += 1       # prediu entitat on no n'hi ha
            elif g_is_entity and not p_is_entity:
                fn += 1       # no prediu entitat on n'hi havia

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {'precision': precision, 'recall': recall, 'f1': f1}

In [ ]:
feature_configs = [
    ('baseline',        FeatureFunc(use_casing=False, use_nums=False, use_prefix=False,
                                    use_suffix=False, use_context=False, use_length=False)),
    ('casing_only',     FeatureFunc(use_nums=False, use_prefix=False,
                                    use_suffix=False, use_context=False, use_length=False)),
    ('full',            FeatureFunc()),  # tot a True per defecte
    ('no_context',      FeatureFunc(use_context=False)),
    ('prefix_suffix',   FeatureFunc(use_casing=False, use_nums=False,
                                    use_context=False, use_length=False)),
]

import pandas as pd
results = []

for coding in ['BIO', 'IO', 'BIOW', 'BIOEW']:
    train_esp = convert_coding(esp_train_net, coding)
    train_ned = convert_coding(ned_train_net, coding)
    val_esp   = convert_coding(esp_val_net,   coding)
    val_ned   = convert_coding(ned_val_net,   coding)

    for feat_name, feat_func in feature_configs:
        for lang, train, val in [('ESP', train_esp, val_esp),
                                  ('NED', train_ned, val_ned)]:

            model = nltk.tag.CRFTagger(feature_func=feat_func)
            model.train(train, os.path.join(models_dir, f'crf_{coding}_{feat_name}_{lang}.mdl'))

            metrics = evaluate(model, val)
            print(f"{coding} | {feat_name} | {lang} => "
                  f"P={metrics['precision']:.4f} "
                  f"R={metrics['recall']:.4f} "
                  f"F1={metrics['f1']:.4f}")
            results.append({'coding': coding, 'features': feat_name,
                            'lang': lang, **metrics})

df = pd.DataFrame(results)
print(df.pivot_table(index=['coding', 'features'], columns='lang', values='f1'))

KeyboardInterrupt: 

In [20]:
feature_configs = [
    ('baseline',        FeatureFunc(use_casing=False, use_nums=False, use_prefix=False,
                                    use_suffix=False, use_context=False, use_length=False))
]

import pandas as pd
results = []

for coding in ['BIO']:
    train_esp = convert_coding(esp_train_net, coding)
    train_ned = convert_coding(ned_train_net, coding)
    val_esp   = convert_coding(esp_val_net,   coding)
    val_ned   = convert_coding(ned_val_net,   coding)

    for feat_name, feat_func in feature_configs:
        for lang, train, val in [('ESP', train_esp, val_esp)]:

            model = nltk.tag.CRFTagger(feature_func=feat_func)
            model.train(train, os.path.join(models_dir, f'crf_{coding}_{feat_name}_{lang}.mdl'))

            metrics = evaluate(model, val)
            print(f"{coding} | {feat_name} | {lang} => "
                  f"P={metrics['precision']:.4f} "
                  f"R={metrics['recall']:.4f} "
                  f"F1={metrics['f1']:.4f}")
            results.append({'coding': coding, 'features': feat_name,
                            'lang': lang, **metrics})

df = pd.DataFrame(results)
print(df.pivot_table(index=['coding', 'features'], columns='lang', values='f1'))

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\UPC\\GIA - Quart Semestre\\PLH - Processament del Llenguatge Humà\\Practica3-PLH\\Practica3_PLH\\models\\crf_BIO_baseline_ESP.mdl'

In [ ]:
# Triem la millor configuració per cada llengua segons val
for lang in ['ESP', 'NED']:
    best = df[df['lang'] == lang].sort_values('accuracy', ascending=False).iloc[0]
    print(f"\nMillor configuració per {lang}: coding={best['coding']}, features={best['features']}")

    # Ara entrenem amb train i avaluem amb TEST (només aquest cop!)
    best_feat_func = dict(feature_configs)[best['features']]
    train_data = convert_coding(esp_train_net if lang == 'ESP' else ned_train_net, best['coding'])
    test_data  = convert_coding(esp_test_net  if lang == 'ESP' else ned_test_net,  best['coding'])

    model = nltk.tag.CRFTagger(feature_func=best_feat_func)
    model.train(train_data, f'/tmp/crf_best_{lang}.mdl')
    test_acc = model.accuracy(test_data)
    print(f"Accuracy final en TEST per {lang}: {test_acc:.4f}")

### Fer la classe de Feature Funciton + Mirar accuracy en funció de entitats i recall i tal